# 02 · Logical model design (profile → star schema proposal)

Reads the output of `01_lakehouse_data_profiling` and proposes a **dimensional (star-schema) logical model**:

1. Classifies every profiled table as `fact`, `dimension`, `bridge`, or `standalone` using the profile
   (row counts, measure / identifier / descriptive column mix, and which tables are referenced by others).
2. Picks a natural key for each dimension, flags sources whose keys need de-duplication, and detects
   **snowflakes** (dimension → dimension links) so they can be flattened.
3. Maps every fact foreign key to a dimension, turns every fact date column into a **role-playing** link to a
   shared `dim_date`, and proposes additive measures with sensible format strings.
4. Emits a **bus matrix**, a Mermaid ER diagram, a review document, and a machine-readable
   `model_spec.json` that `03_build_dim_fact_model` and `04_semantic_model_deploy` consume.

Nothing here is final: the heuristics get you a defensible first draft in minutes; the `TABLE_ROLE_OVERRIDES`
and `NATURAL_KEY_OVERRIDES` parameters (or editing `model_spec.json` directly) capture the decisions you make
with the business.

In [ ]:
# PARAMETERS — override from a pipeline or notebookutils.notebook.run()
LAKEHOUSE_ROOT = ""            # abfss://<workspace-id>@onelake.dfs.fabric.microsoft.com/<lakehouse-id> ; "" = default lakehouse
PROFILING_SCHEMA = "profiling" # schema (or table prefix) used by notebook 01
RUN_ID = ""                    # "" = latest profiling run
MODEL_NAME = "Lakehouse Model" # semantic model / spec name
GOLD_SCHEMA = "gold"           # target schema for dim_/fact_ tables (prefix on non-schema lakehouses)
SPEC_PATH = "Files/model/model_spec.json"   # relative to the lakehouse root
TABLE_ROLE_OVERRIDES = ""      # e.g. "orders=fact,customers=dimension,order_items=fact,audit_log=ignore"
NATURAL_KEY_OVERRIDES = ""     # e.g. "customers=customer_id,products=sku"
FLATTEN_SNOWFLAKES = True      # fold dimension→dimension parents into the child dimension
MIN_CONTAINMENT = 0.95         # relationship candidates below this are listed as "unresolved" rather than modelled
DATE_DIM_NAME = "dim_date"

In [ ]:
import re, json, datetime as dt, statistics
from pyspark.sql import functions as F


def _parse_map(s):
    out = {}
    for part in (s or "").split(","):
        if "=" in part:
            k, v = part.split("=", 1)
            out[k.strip().lower()] = v.strip()
    return out


def _lakehouse_root():
    if LAKEHOUSE_ROOT:
        return LAKEHOUSE_ROOT.rstrip("/")
    ctx = notebookutils.runtime.context
    ws, lh = ctx.get("defaultLakehouseWorkspaceId"), ctx.get("defaultLakehouseId")
    if not lh:
        raise RuntimeError("Attach a default Lakehouse to this notebook or set LAKEHOUSE_ROOT.")
    return f"abfss://{ws}@onelake.dfs.fabric.microsoft.com/{lh}"


ROOT = _lakehouse_root()
SCHEMA_ENABLED = notebookutils.fs.exists(f"{ROOT}/Tables/{PROFILING_SCHEMA}/table_profile")


def profiling_path(name):
    return f"{ROOT}/Tables/{PROFILING_SCHEMA}/{name}" if SCHEMA_ENABLED else f"{ROOT}/Tables/{PROFILING_SCHEMA}_{name}"


tp_all = spark.read.format("delta").load(profiling_path("table_profile"))
RUN_ID = RUN_ID or tp_all.agg(F.max("run_id")).first()[0]
if not RUN_ID:
    raise RuntimeError("No profiling runs found — run 01_lakehouse_data_profiling first.")
tables = [r.asDict() for r in tp_all.where(F.col("run_id") == RUN_ID).collect()]
columns = [r.asDict() for r in spark.read.format("delta").load(profiling_path("column_profile")).where(F.col("run_id") == RUN_ID).collect()]
rels = [r.asDict() for r in spark.read.format("delta").load(profiling_path("relationship_candidates")).where(F.col("run_id") == RUN_ID).collect()]

cols_of = {}
for c in columns:
    cols_of.setdefault(c["table_name"], []).append(c)
for v in cols_of.values():
    v.sort(key=lambda x: x["ordinal"])
role_overrides, key_overrides = _parse_map(TABLE_ROLE_OVERRIDES), _parse_map(NATURAL_KEY_OVERRIDES)
print(f"Profiling run {RUN_ID}: {len(tables)} tables, {len(columns)} columns, {sum(r['is_candidate'] for r in rels)} relationship candidates")

## Step 1 — Classify tables

| Signal | Points toward |
|---|---|
| ≥ 1 measure column, ≥ 1 identifier pointing at another table, a date column, above-median row count | **fact** |
| Referenced by other tables, has a (near-)unique key, several descriptive (category / text) columns, below-median row count | **dimension** |
| Only foreign keys, no measures, no descriptive columns | **bridge** (many-to-many resolver) |
| Nothing links to it and it links to nothing | **standalone** — review manually |

Scores are shown so you can see *why* a table landed where it did. Override with `TABLE_ROLE_OVERRIDES`.

In [ ]:
for r in rels:  # profiles written by older versions of notebook 01 lack the row-weighted columns
    r.setdefault("containment_rows", r["containment"]); r.setdefault("orphan_rows", r["orphan_distinct"])
cand_rels = [r for r in rels if r["is_candidate"] and r["containment_rows"] >= MIN_CONTAINMENT]
# keep the best parent per child column (highest containment, prefer unique parent keys)
best = {}
for r in sorted(cand_rels, key=lambda r: (-r["containment_rows"], not r["parent_key_is_unique"])):
    best.setdefault((r["child_table"], r["child_column"]), r)
cand_rels = list(best.values())
referenced_by = {}
fk_out = {}
for r in cand_rels:
    referenced_by.setdefault(r["parent_table"], set()).add(r["child_table"])
    fk_out.setdefault(r["child_table"], set()).add(r["parent_table"])

row_counts = [t["row_count"] for t in tables if t["row_count"] > 0]
median_rows = statistics.median(row_counts) if row_counts else 0


def natural_key_for(t):
    name = t["table_name"]
    if name.lower() in key_overrides:
        return key_overrides[name.lower()], True
    if t["candidate_pk"]:
        return t["candidate_pk"], True
    near = [c for c in cols_of.get(name, []) if c["semantic_role"] == "identifier" and c["uniqueness_ratio"] >= 0.95]
    near.sort(key=lambda c: -c["uniqueness_ratio"])
    if near:
        return near[0]["column_name"], False
    return None, False


classified = {}
for t in tables:
    name = t["table_name"]
    cols = cols_of.get(name, [])
    fk_cols = {r["child_column"] for r in cand_rels if r["child_table"] == name}
    measures = [c for c in cols if c["semantic_role"] == "measure" and c["column_name"] not in fk_cols]
    descriptive = [c for c in cols if c["semantic_role"] in ("category", "text")]
    dates = [c for c in cols if c["semantic_role"] == "date"]
    nk, nk_unique = natural_key_for(t)
    n_out, n_in = len(fk_out.get(name, ())), len(referenced_by.get(name, ()))
    # A table that links to ≥2 others AND carries dates behaves like a transaction header even when children reference it
    # (orders ← order_items). Otherwise being referenced is strong dimension evidence.
    header_like = n_out >= 2 and bool(dates)
    if n_out == 0 and not dates:
        fact_score = 0  # nothing to slice it by → cannot be a fact, whatever numeric columns it carries
    else:
        fact_score = 2 * min(n_out, 3) + 2 * min(len(measures), 3) + (1 if dates else 0) + (1 if t["row_count"] > median_rows else 0) - (0 if header_like else (3 if n_in else 0))
    dim_score = ((1 if header_like else 3) if n_in else 0) + (1 if nk else 0) + (1 if len(descriptive) >= 2 else 0) + (1 if t["row_count"] <= median_rows else 0) - (2 if len(measures) >= 2 and n_out >= 2 else 0)
    if n_out >= 2 and not measures and not descriptive and not n_in:
        role = "bridge"
    elif fact_score >= 3 and fact_score > dim_score:
        role = "fact"
    elif dim_score >= 2:
        role = "dimension"
    elif n_out == 0 and n_in == 0:
        role = "standalone"
    else:
        role = "dimension" if nk else "fact"
    if name.lower() in role_overrides:
        role = role_overrides[name.lower()]
    classified[name] = {"table": t, "role": role, "fact_score": fact_score, "dim_score": dim_score, "natural_key": nk,
                        "natural_key_unique": nk_unique, "measures": measures, "descriptive": descriptive, "dates": dates,
                        "fk_cols": fk_cols, "referenced_by": sorted(referenced_by.get(name, ())), "references": sorted(fk_out.get(name, ()))}

print(f"{'table':30s} {'role':11s} {'fact':>4} {'dim':>4} {'rows':>10}  key                 refs→        ←referenced by")
for name, c in sorted(classified.items(), key=lambda kv: (kv[1]['role'], kv[0])):
    print(f"{name:30s} {c['role']:11s} {c['fact_score']:>4} {c['dim_score']:>4} {c['table']['row_count']:>10,}  "
          f"{(c['natural_key'] or '-') + ('' if c['natural_key_unique'] else ' (dupes)'):20s}{','.join(c['references']):13s}{','.join(c['referenced_by'])}")

## Step 2 — Dimensions, facts, snowflakes, date roles, measures

In [ ]:
def _singular(word):
    w = word.lower()
    for pre in ("dim_", "fact_", "tbl_", "stg_", "raw_", "src_", "bronze_", "silver_", "t_", "v_"):
        if w.startswith(pre):
            w = w[len(pre):]
    if w.endswith("ies"):
        return w[:-3] + "y"
    if w.endswith(("ses", "xes", "zes", "ches", "shes")):
        return w[:-2]
    if w.endswith("s") and not w.endswith("ss"):
        return w[:-1]
    return w


def _strip_prefix(word):
    w = word.lower()
    for pre in ("dim_", "fact_", "tbl_", "stg_", "raw_", "src_", "bronze_", "silver_", "t_", "v_"):
        if w.startswith(pre):
            return w[len(pre):]
    return w


def _role_name(child_col, parent_table):
    """Role prefix when a fact links twice to the same dimension, e.g. ship_customer_id → 'ship'."""
    base = _singular(parent_table)
    c = child_col.lower()
    for suf in ("_id", "id", "_key", "key", "_code", "code", "_sk", "_fk"):
        if c.endswith(suf):
            c = c[: -len(suf)]
            break
    c = c.rstrip("_")
    return "" if c in (base, base.replace("_", ""), "") else c


def _format_for(colname):
    n = colname.lower()
    if re.search(r"(pct|percent|rate|ratio|margin)", n):
        return "0.0%"
    if re.search(r"(amount|amt|price|cost|revenue|sales|total|fee|tax|discount|balance|value)", n):
        return "#,##0.00"
    return "#,##0"


def _title(s):
    return " ".join(w.capitalize() for w in re.split(r"[_\s]+", s) if w)


table_meta = {t["table_name"]: t for t in tables}
dims, facts, bridges, unresolved, notes = {}, {}, {}, [], []
dim_name_of = {}

# --- dimensions -----------------------------------------------------------------------------
for name, c in classified.items():
    if c["role"] != "dimension":
        continue
    if not c["natural_key"]:
        unresolved.append({"table": name, "issue": "dimension has no usable natural key (no unique or near-unique identifier column)"})
        continue
    dname = "dim_" + _singular(name)
    dim_name_of[name] = dname
    attrs = [x["column_name"] for x in cols_of[name] if x["column_name"] != c["natural_key"]]
    dims[dname] = {"name": dname, "source_schema": table_meta[name]["schema_name"], "source_table": name,
                   "source_row_count": table_meta[name]["row_count"], "natural_key": [c["natural_key"]],
                   "natural_key_is_unique": c["natural_key_unique"], "surrogate_key": _singular(name) + "_key",
                   "attributes": attrs, "scd_type": 1, "snowflake_parents": [], "referenced_by": c["referenced_by"]}
    if not c["natural_key_unique"]:
        notes.append(f"{dname}: source {name}.{c['natural_key']} has duplicate values — build de-duplicates (keeps one row per key); confirm the rule with the data owner.")

# --- snowflakes: dimension → dimension ------------------------------------------------------
for r in cand_rels:
    ch, pa = r["child_table"], r["parent_table"]
    if classified.get(ch, {}).get("role") == "dimension" and classified.get(pa, {}).get("role") == "dimension" and ch in dim_name_of and pa in dim_name_of:
        parent_attrs = [x["column_name"] for x in cols_of[pa] if x["column_name"] != r["parent_column"]]
        entry = {"table": pa, "schema": table_meta[pa]["schema_name"], "child_column": r["child_column"], "parent_column": r["parent_column"],
                 "attributes": parent_attrs, "prefix": _singular(pa) + "_"}
        if FLATTEN_SNOWFLAKES:
            dims[dim_name_of[ch]]["snowflake_parents"].append(entry)
            dims[dim_name_of[ch]]["attributes"] = [a for a in dims[dim_name_of[ch]]["attributes"] if a != r["child_column"]] + [r["child_column"]]
            notes.append(f"{dim_name_of[ch]}: flattened snowflake parent {pa} (columns prefixed '{entry['prefix']}'); {dim_name_of[pa]} is kept only if a fact references it directly.")
        else:
            notes.append(f"{dim_name_of[ch]} → {dim_name_of[pa]} is a snowflake; Direct Lake works better with flattened dimensions.")

# --- facts ------------------------------------------------------------------------------------
date_ranges = []
fact_name_of = {n: ("fact_" if c["role"] == "fact" else "bridge_") + _strip_prefix(n) for n, c in classified.items() if c["role"] in ("fact", "bridge")}
header_links = {}  # line fact → [(header source table, child col, header col)]

for name, c in classified.items():
    if c["role"] not in ("fact", "bridge"):
        continue
    fname = fact_name_of[name]
    fks, seen_dims = [], {}
    for r in cand_rels:
        if r["child_table"] != name:
            continue
        pa = r["parent_table"]
        if pa not in dim_name_of:
            if pa in fact_name_of:
                header_links.setdefault(name, []).append({"header_table": pa, "header_fact": fact_name_of[pa], "child_column": r["child_column"], "header_column": r["parent_column"]})
                notes.append(f"{fname}.{r['child_column']} → {pa} is a header/line link: {fname} inherits {fact_name_of[pa]}'s dimension and date keys so line measures can be sliced by header attributes.")
            else:
                unresolved.append({"table": name, "column": r["child_column"], "issue": f"references {pa}, which is not modelled as a dimension"})
            continue
        dname = dim_name_of[pa]
        role = _role_name(r["child_column"], pa)
        seen_dims.setdefault(dname, 0)
        seen_dims[dname] += 1
        fks.append({"column": r["child_column"], "dimension": dname, "dimension_natural_key": r["parent_column"], "role": role,
                    "surrogate_key_column": (role + "_" if role else "") + dims[dname]["surrogate_key"],
                    "containment_rows": r["containment_rows"], "orphan_rows": r["orphan_rows"], "orphan_distinct": r["orphan_distinct"], "inherited_from": None})
        if r["orphan_rows"]:
            notes.append(f"{fname}.{r['child_column']}: {r['orphan_rows']:,} rows ({r['orphan_distinct']} distinct values) have no match in {dname} — they will map to the Unknown member (-1).")
    for d, n in seen_dims.items():
        if n > 1:
            notes.append(f"{fname} links to {d} {n} times (role-playing dimension); only the first relationship will be active in the semantic model.")
    dates = []
    for i, dcol in enumerate(c["dates"]):
        base = re.sub(r"(_date|_dt|date|_ts|_timestamp|timestamp)$", "", dcol["column_name"].lower()).strip("_") or "date"
        dates.append({"column": dcol["column_name"], "data_type": dcol["data_type"], "date_key_column": f"{base}_date_key",
                      "dimension": DATE_DIM_NAME, "role": base, "active": i == 0, "inherited_from": None})
        if dcol["min_value"] and dcol["max_value"]:
            date_ranges.append((dcol["min_value"][:10], dcol["max_value"][:10]))
    fk_cols = {f["column"] for f in fks} | {d["column"] for d in dates} | {h["child_column"] for h in header_links.get(name, [])}
    measures = [{"column": m["column_name"], "data_type": m["data_type"], "aggregation": "sum", "measure_name": "Total " + _title(m["column_name"]),
                 "format_string": _format_for(m["column_name"])} for m in c["measures"] if m["column_name"] not in fk_cols]
    nk = c["natural_key"]
    degenerate = [x["column_name"] for x in cols_of[name] if x["column_name"] not in fk_cols and x["column_name"] not in {m["column"] for m in measures}]
    degenerate += [h["child_column"] for h in header_links.get(name, [])]  # keep the header id on the line fact (drill-through)
    entry = {"name": fname, "source_schema": table_meta[name]["schema_name"], "source_table": name, "source_row_count": table_meta[name]["row_count"],
             "grain": f"one row per {name} row" + (f" (unique on {nk})" if nk and c["natural_key_unique"] else ""),
             "natural_key": [nk] if nk else [], "foreign_keys": fks, "date_columns": dates, "measures": measures,
             "degenerate_columns": degenerate, "header_links": header_links.get(name, []), "row_count_measure": "Row Count " + _title(_strip_prefix(name))}
    (facts if c["role"] == "fact" else bridges)[fname] = entry

# --- header/line inheritance: a line fact gets the header fact's dimension + date keys ---------
all_facts = {**facts, **bridges}
for line_src, links in header_links.items():
    line = all_facts[fact_name_of[line_src]]
    for h in links:
        header = all_facts[h["header_fact"]]
        have_fk = {(fk["dimension"], fk["role"]) for fk in line["foreign_keys"]}
        for fk in header["foreign_keys"]:
            if (fk["dimension"], fk["role"]) in have_fk:
                continue
            line["foreign_keys"].append({**fk, "inherited_from": {"header_fact": header["name"], "header_table": h["header_table"],
                                                                 "child_column": h["child_column"], "header_column": h["header_column"]}})
        have_dates = {d["role"] for d in line["date_columns"]}
        for d in header["date_columns"]:
            if d["role"] in have_dates:
                continue
            line["date_columns"].append({**d, "active": not line["date_columns"] and d["active"],
                                         "inherited_from": {"header_fact": header["name"], "header_table": h["header_table"],
                                                            "child_column": h["child_column"], "header_column": h["header_column"]}})
for f in all_facts.values():
    if not f["foreign_keys"] and not f["date_columns"]:
        unresolved.append({"table": f["source_table"], "issue": "classified as fact but has no dimension or date links — check TABLE_ROLE_OVERRIDES"})

# drop dimensions that were flattened into a child and are referenced by no fact
if FLATTEN_SNOWFLAKES:
    used = {fk["dimension"] for f in list(facts.values()) + list(bridges.values()) for fk in f["foreign_keys"]}
    flattened = {p["table"] for d in dims.values() for p in d["snowflake_parents"]}
    for src in flattened:
        dname = dim_name_of[src]
        if dname not in used:
            dims.pop(dname, None)
            notes.append(f"{dname} removed from the model: fully absorbed into its child dimension and referenced by no fact.")

for name, c in classified.items():
    if c["role"] == "standalone":
        unresolved.append({"table": name, "issue": "standalone table — no relationships found; include via overrides or leave out"})
    elif c["role"] == "ignore":
        notes.append(f"{name} ignored via TABLE_ROLE_OVERRIDES.")

# --- date dimension range ----------------------------------------------------------------------
if date_ranges:
    start = min(d[0] for d in date_ranges)
    end = max(d[1] for d in date_ranges)
    start = f"{int(start[:4])}-01-01"
    end = f"{int(end[:4]) + 1}-12-31"
else:
    start, end = f"{dt.date.today().year - 5}-01-01", f"{dt.date.today().year + 1}-12-31"
date_dim = {"name": DATE_DIM_NAME, "surrogate_key": "date_key", "start": start, "end": end, "fiscal_year_start_month": 1}

print(f"Dimensions: {list(dims)}")
print(f"Facts:      {list(facts)}")
print(f"Bridges:    {list(bridges)}")
print(f"Date dim:   {DATE_DIM_NAME} {start} → {end}")
print(f"Unresolved: {len(unresolved)}   Notes: {len(notes)}")

## Step 3 — Bus matrix, ER diagram, review document, `model_spec.json`

In [ ]:
all_dims = [DATE_DIM_NAME] + sorted(dims)
bus_rows = []
for f in list(facts.values()) + list(bridges.values()):
    row = {"fact": f["name"], "grain": f["grain"]}
    for d in all_dims:
        n = sum(1 for fk in f["foreign_keys"] if fk["dimension"] == d) + (len(f["date_columns"]) if d == DATE_DIM_NAME else 0)
        row[d] = "X" if n == 1 else (f"X×{n}" if n > 1 else "")
    bus_rows.append(row)

bus_md = "| Fact | " + " | ".join(all_dims) + " |\n|---|" + "---|" * len(all_dims) + "\n"
for r in bus_rows:
    bus_md += f"| {r['fact']} | " + " | ".join(r[d] for d in all_dims) + " |\n"
print("BUS MATRIX\n" + bus_md)

mermaid = ["erDiagram"]
mermaid.append(f"    {DATE_DIM_NAME} {{ int date_key PK \n        date full_date \n        int year \n        string month_name }}")
type_of = {(c["table_name"], c["column_name"]): re.sub(r"[^a-z0-9_]", "", c["data_type"].split("(")[0]) for c in columns}
for d in dims.values():
    src = d["source_table"]
    lines = [f"        bigint {d['surrogate_key']} PK"] + [f"        {type_of.get((src, a), 'string')} {a}" for a in d["natural_key"]]
    lines += [f"        {type_of.get((src, a), 'string')} {a}" for a in d["attributes"][:6]]
    if len(d["attributes"]) > 6:
        lines.append(f"        string more_{len(d['attributes']) - 6}_columns")
    mermaid.append(f"    {d['name']} {{\n" + "\n".join(lines) + "\n    }")
for f in list(facts.values()) + list(bridges.values()):
    lines = [f"        bigint {fk['surrogate_key_column']} FK" for fk in f["foreign_keys"]] + [f"        int {dc['date_key_column']} FK" for dc in f["date_columns"]]
    lines += [f"        {type_of.get((f['source_table'], m['column']), 'double')} {m['column']}" for m in f["measures"][:6]]
    mermaid.append(f"    {f['name']} {{\n" + "\n".join(lines) + "\n    }")
    for fk in f["foreign_keys"]:
        mermaid.append(f"    {fk['dimension']} ||--o{{ {f['name']} : \"{fk['role'] or fk['column']}\"")
    for dc in f["date_columns"]:
        mermaid.append(f"    {DATE_DIM_NAME} ||--o{{ {f['name']} : \"{dc['role']}\"")
mermaid_txt = "\n".join(mermaid)

spec = {
    "model_name": MODEL_NAME, "generated_at": dt.datetime.utcnow().isoformat() + "Z", "profiling_run_id": RUN_ID,
    "lakehouse_root": ROOT, "schema_enabled": SCHEMA_ENABLED, "gold_schema": GOLD_SCHEMA, "unknown_member_key": -1,
    "date_dimension": date_dim, "dimensions": list(dims.values()), "facts": list(facts.values()), "bridges": list(bridges.values()),
    "bus_matrix": bus_rows, "classification": {n: {"role": c["role"], "fact_score": c["fact_score"], "dim_score": c["dim_score"]} for n, c in classified.items()},
    "unresolved": unresolved, "notes": notes, "mermaid": mermaid_txt,
}
spec_json = json.dumps(spec, indent=2, default=str)
notebookutils.fs.put(f"{ROOT}/{SPEC_PATH}", spec_json, True)
notebookutils.fs.put(f"{ROOT}/{SPEC_PATH.rsplit('.', 1)[0]}.mmd", mermaid_txt, True)

review = [f"# Logical model review — {MODEL_NAME}", f"Generated {spec['generated_at']} from profiling run `{RUN_ID}`", "",
          "## Bus matrix", bus_md, "## Dimensions"]
for d in dims.values():
    review.append(f"- **{d['name']}** ← `{d['source_table']}` ({d['source_row_count']:,} rows), natural key `{','.join(d['natural_key'])}`"
                  + ("" if d["natural_key_is_unique"] else " **(duplicates in source)**")
                  + (f", flattens {', '.join(p['table'] for p in d['snowflake_parents'])}" if d["snowflake_parents"] else ""))
review.append(f"- **{DATE_DIM_NAME}** generated {start} → {end}")
review.append("\n## Facts")
for f in list(facts.values()) + list(bridges.values()):
    review.append(f"- **{f['name']}** ← `{f['source_table']}` ({f['source_row_count']:,} rows); grain: {f['grain']}")
    for fk in f["foreign_keys"]:
        via = f" via {fk['inherited_from']['header_fact']}" if fk.get("inherited_from") else ""
        review.append(f"  - `{fk['column']}`{via} → {fk['dimension']}" + (f" (role: {fk['role']})" if fk["role"] else "") + f", row containment {fk['containment_rows']:.1%}")
    for dc in f["date_columns"]:
        via = f" via {dc['inherited_from']['header_fact']}" if dc.get("inherited_from") else ""
        review.append(f"  - `{dc['column']}`{via} → {DATE_DIM_NAME} as `{dc['date_key_column']}`" + (" (active)" if dc["active"] else " (inactive, use USERELATIONSHIP)"))
    for m in f["measures"]:
        review.append(f"  - measure **{m['measure_name']}** = SUM(`{m['column']}`) format `{m['format_string']}`")
review.append("\n## Notes")
review += [f"- {n}" for n in notes]
review.append("\n## Unresolved — decide before building")
review += [f"- **{u['table']}**" + (f".`{u['column']}`" if u.get("column") else "") + f": {u['issue']}" for u in unresolved] or ["- none"]
review.append("\n## ER diagram (Mermaid)\n```mermaid\n" + mermaid_txt + "\n```")
notebookutils.fs.put(f"{ROOT}/{SPEC_PATH.rsplit('/', 1)[0]}/logical_model_review.md", "\n".join(review), True)

print(f"Wrote {ROOT}/{SPEC_PATH}")
print(f"Wrote {ROOT}/{SPEC_PATH.rsplit('/', 1)[0]}/logical_model_review.md (+ .mmd diagram)")
print()
print("\n".join(review))

### Next step
Review the notes and unresolved items above. Fix decisions with `TABLE_ROLE_OVERRIDES` / `NATURAL_KEY_OVERRIDES` and re-run,
or edit `Files/model/model_spec.json` directly. Then run **`03_build_dim_fact_model`**.